# 🧪 Part 3: A/B Testing with Python

Welcome! Now that you are comfortable with basic Python and Pandas, we will learn **A/B Testing** — the main tool data analysts use to decide "is version B actually better than version A, or did we just get lucky?"

### Key vocabulary
- **Control group (A)**: the "current" version — the baseline we compare against.
- **Treatment group (B)**: the "new" version we want to test.
- **Metric**: the number we measure and compare (e.g. conversion rate, average time on page, revenue).
- **Null Hypothesis (H0)**: "There is NO real difference between the groups. Any difference we see is just random noise."
- **Alternative Hypothesis (H1)**: "There IS a real difference between the groups."
- **P-value**: the probability of seeing a difference this large (or larger) if H0 were actually true. A small p-value means the difference is unlikely to be random noise.
- **Significance level (alpha)**: the threshold we compare the p-value to. We will use the standard `alpha = 0.05` (5%) throughout this notebook.

### What we will do in this notebook
1. Build a simulated A/B test dataset
2. Explore and visualize the two groups
3. Calculate conversion rates
4. Formulate hypotheses and check assumptions
5. Run a **two-proportion z-test** (for binary metrics like "did they convert?")
6. Run a **t-test** (for continuous metrics like "time spent" or "revenue")
7. Compute confidence intervals and effect sizes
8. Walk through two full **business case studies**
9. Learn about common pitfalls (peeking, small samples) and how to calculate the sample size you need **before** running a test


## 1. Setup: Import Libraries

- **pandas / numpy**: to build and manipulate our dataset
- **scipy.stats**: general-purpose statistics (t-tests, Q-Q plots)
- **statsmodels**: specialized A/B testing functions (proportion z-tests, confidence intervals, power/sample-size analysis)
- **matplotlib / seaborn**: to visualize our groups

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Statistics
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportions_confint, proportion_effectsize
from statsmodels.stats.power import NormalIndPower, TTestIndPower

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
np.random.seed(42)  # makes our "random" data reproducible

print("Libraries loaded successfully!")

## 2. Generate a Sample A/B Test Dataset

Imagine we ran an experiment on our website. Half of the visitors saw the **control** (current) version, and half saw the **treatment** (new) version. For every visitor we recorded:
- `converted`: 1 if they made a purchase, 0 if they didn't (our **binary** metric)
- `time_spent`: minutes spent on the page (our **continuous** metric)

In real life you would load this from a database or CSV. Here we **simulate** it, so we can practice on data with a known, realistic effect.

In [ ]:
n_control = 2000
n_treatment = 2000

# These are the TRUE, hidden parameters we're simulating.
# In a real experiment you would NOT know these values in advance!
true_conversion_control = 0.10
true_conversion_treatment = 0.12

true_time_mean_control = 5.0     # average minutes spent on page
true_time_mean_treatment = 5.3
true_time_std = 1.5

control = pd.DataFrame({
    "user_id": range(1, n_control + 1),
    "group": "control",
    "converted": np.random.binomial(n=1, p=true_conversion_control, size=n_control),
    "time_spent": np.round(np.random.normal(true_time_mean_control, true_time_std, n_control).clip(min=0.1), 2)
})

treatment = pd.DataFrame({
    "user_id": range(n_control + 1, n_control + n_treatment + 1),
    "group": "treatment",
    "converted": np.random.binomial(n=1, p=true_conversion_treatment, size=n_treatment),
    "time_spent": np.round(np.random.normal(true_time_mean_treatment, true_time_std, n_treatment).clip(min=0.1), 2)
})

df = pd.concat([control, treatment], ignore_index=True)
df.head()

## 3. Exploratory Data Analysis (EDA)

Before running any test, always look at your data first: how many users per group, what do the numbers look like, and is anything missing?

In [ ]:
# How many users are in each group? (Ideally this should be roughly 50/50)
print(df["group"].value_counts())
print()

# Basic statistics for the numeric columns, split by group
print(df.groupby("group")[["converted", "time_spent"]].describe())
print()

# Always check for missing data before running any statistical test
print("Missing values per column:")
print(df.isna().sum())

## 4. Visualize Group Distributions

A picture is worth a thousand p-values. Let's compare the two groups visually before doing any formal test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left plot: conversion rate per group (as a percentage)
conversion_rate_pct = df.groupby("group")["converted"].mean() * 100
sns.barplot(x=conversion_rate_pct.index, y=conversion_rate_pct.values, ax=axes[0])
axes[0].set_ylabel("Conversion rate (%)")
axes[0].set_title("Conversion Rate by Group")

# Right plot: distribution of time spent per group
sns.histplot(data=df, x="time_spent", hue="group", kde=True, stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Time Spent on Page by Group")

plt.tight_layout()
plt.show()

## 5. Calculate Conversion Rates

Let's put the exact numbers we need for the statistical test into one clean table: the number of users, the number of conversions, and the conversion rate for each group.

In [ ]:
conversion_summary = df.groupby("group")["converted"].agg(users="count", conversions="sum")
conversion_summary["conversion_rate"] = conversion_summary["conversions"] / conversion_summary["users"]
conversion_summary

## 6. Formulate Hypotheses

Before testing, we must write down — in advance — what we are trying to prove. This prevents us from changing our conclusion after seeing the data.

**For the conversion rate (binary metric):**
- $H_0$ (null): $p_{control} = p_{treatment}$ — the conversion rate is the same in both groups.
- $H_1$ (alternative): $p_{control} \neq p_{treatment}$ — the conversion rates are different.

**For time spent on page (continuous metric):**
- $H_0$ (null): $\mu_{control} = \mu_{treatment}$ — the average time spent is the same in both groups.
- $H_1$ (alternative): $\mu_{control} \neq \mu_{treatment}$ — the average time spent is different.

Note: these are **two-sided** hypotheses (we test for "different", not specifically "bigger" or "smaller"). This is the safest default choice.

## 7. Check Statistical Assumptions

Every statistical test has assumptions. If we skip this step, we might get a p-value we can't trust. Here's what we check:

- **For the z-test on proportions**: each group needs at least ~5 conversions AND at least ~5 non-conversions (rule of thumb: $n \times p \geq 5$ and $n \times (1-p) \geq 5$). This ensures the sampling distribution is well approximated by a normal curve.
- **For the t-test on a continuous metric**: the *data itself* doesn't need to be perfectly normal, but with a large enough sample size (roughly n > 30 per group), the *average* will be approximately normal thanks to the **Central Limit Theorem**. A Q-Q plot helps us sanity-check this visually.

In [ ]:
# Check: do we have enough conversions AND non-conversions in each group?
for grp in ["control", "treatment"]:
    sub = df[df["group"] == grp]
    n = len(sub)
    p = sub["converted"].mean()
    print(f"{grp}: n={n}, conversion_rate={p:.3f}, n*p={n*p:.1f}, n*(1-p)={n*(1-p):.1f}  -> both should be >= 5")

# Visual check: is time_spent roughly normal in each group?
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, grp in zip(axes, ["control", "treatment"]):
    stats.probplot(df[df["group"] == grp]["time_spent"], dist="norm", plot=ax)
    ax.set_title(f"Q-Q plot: {grp}")
plt.tight_layout()
plt.show()

## 8. Perform a Two-Proportion Z-Test

Now for the real test! A **two-proportion z-test** compares two conversion rates and tells us how surprising the observed difference would be if `H0` were true.

We use `statsmodels`' `proportions_ztest()`, which needs:
- `count`: the number of "successes" (conversions) in each group
- `nobs`: the total number of observations (users) in each group

In [ ]:
successes = conversion_summary["conversions"].values  # [control_conversions, treatment_conversions]
nobs = conversion_summary["users"].values              # [control_n, treatment_n]

z_stat, p_value_conversion = proportions_ztest(count=successes, nobs=nobs, alternative="two-sided")

print(f"Z-statistic: {z_stat:.3f}")
print(f"P-value: {p_value_conversion:.4f}")

## 9. Perform a T-Test for the Continuous Metric

For `time_spent`, we use an **independent samples t-test** (`scipy.stats.ttest_ind`). We set `equal_var=False` (this is called **Welch's t-test**) — a safe default that doesn't assume both groups have the exact same variance.

In [ ]:
control_time = df.loc[df["group"] == "control", "time_spent"]
treatment_time = df.loc[df["group"] == "treatment", "time_spent"]

t_stat, p_value_time = stats.ttest_ind(control_time, treatment_time, equal_var=False)

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value_time:.4f}")

## 10. Calculate Confidence Intervals

A p-value only tells us "is there a difference?". A **confidence interval (CI)** tells us the likely **range** of that difference. A 95% CI means: "if we repeated this experiment many times, 95% of the intervals we'd compute would contain the true difference."

In [ ]:
# --- CI for each group's conversion rate individually ---
ci_control = proportions_confint(count=successes[0], nobs=nobs[0], alpha=0.05)
ci_treatment = proportions_confint(count=successes[1], nobs=nobs[1], alpha=0.05)
print(f"Control 95% CI: ({ci_control[0]:.4f}, {ci_control[1]:.4f})")
print(f"Treatment 95% CI: ({ci_treatment[0]:.4f}, {ci_treatment[1]:.4f})")

# --- 95% CI for the DIFFERENCE in conversion rates (treatment - control) ---
p1 = conversion_summary.loc["control", "conversion_rate"]
p2 = conversion_summary.loc["treatment", "conversion_rate"]
n1, n2 = nobs[0], nobs[1]

diff_conversion = p2 - p1
se_diff_conversion = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
ci_low, ci_high = diff_conversion - 1.96 * se_diff_conversion, diff_conversion + 1.96 * se_diff_conversion

print(f"\nDifference in conversion rate (treatment - control): {diff_conversion:.4f}")
print(f"95% CI for the difference: [{ci_low:.4f}, {ci_high:.4f}]")

# --- 95% CI for the DIFFERENCE in mean time_spent (treatment - control) ---
mean_diff_time = treatment_time.mean() - control_time.mean()
se_diff_time = np.sqrt(control_time.var(ddof=1) / len(control_time) + treatment_time.var(ddof=1) / len(treatment_time))
ci_low_t, ci_high_t = mean_diff_time - 1.96 * se_diff_time, mean_diff_time + 1.96 * se_diff_time

print(f"\nDifference in mean time spent (treatment - control): {mean_diff_time:.4f} minutes")
print(f"95% CI for the difference: [{ci_low_t:.4f}, {ci_high_t:.4f}]")

## 11. Interpret P-Values and Statistical Significance

The decision rule is simple:
- If `p_value < alpha` → **reject H0**. The difference is "statistically significant" (unlikely to be random noise).
- If `p_value >= alpha` → **fail to reject H0**. We don't have enough evidence to say there's a real difference.

⚠️ Important: "fail to reject H0" does **not** mean "H0 is true" — it just means we don't have enough evidence yet (maybe we need more data).

In [ ]:
alpha = 0.05

def interpret(p_value, alpha, metric_name):
    """Print a plain-English conclusion for a hypothesis test."""
    if p_value < alpha:
        print(f"[{metric_name}] p-value = {p_value:.4f} < alpha = {alpha}  -> REJECT H0. The difference IS statistically significant.")
    else:
        print(f"[{metric_name}] p-value = {p_value:.4f} >= alpha = {alpha}  -> FAIL TO REJECT H0. Not enough evidence of a difference.")

interpret(p_value_conversion, alpha, "Conversion rate (z-test)")
interpret(p_value_time, alpha, "Time spent (t-test)")

## 12. Calculate Effect Size

Statistical significance ≠ practical (business) significance! With a large enough sample, even a tiny, meaningless difference can become "statistically significant". **Effect size** measures how *big* the difference actually is.

- **Cohen's d** (for continuous metrics): the difference in means, measured in standard deviations. Rule of thumb: `0.2` = small, `0.5` = medium, `0.8` = large.
- **Cohen's h** (for proportions): the equivalent idea for conversion rates — we'll use this in section 15.

In [ ]:
def cohens_d(group_a, group_b):
    """Cohen's d: standardized difference between two group means (group_b - group_a)."""
    n_a, n_b = len(group_a), len(group_b)
    pooled_std = np.sqrt(((n_a - 1) * group_a.var(ddof=1) + (n_b - 1) * group_b.var(ddof=1)) / (n_a + n_b - 2))
    return (group_b.mean() - group_a.mean()) / pooled_std

d = cohens_d(control_time, treatment_time)
print(f"Cohen's d for time_spent: {d:.3f}")
print("(0.2 = small, 0.5 = medium, 0.8 = large effect)")

### ✍️ Practice 1

A colleague ran a **smaller** experiment and only gives you the summary numbers:
- Control: `n = 500`, `conversions = 40`
- Treatment: `n = 500`, `conversions = 55`

**Your task:**
1. Calculate the conversion rate for each group.
2. Run a two-proportion z-test with `proportions_ztest()`.
3. Use the `interpret()` function to decide if the result is statistically significant at `alpha = 0.05`.

In [ ]:
# Your code here:

## 13. Business Case: E-commerce "Buy Now" Button Color

**Scenario:** An online store currently uses a **red** "Buy Now" button (control). The marketing team believes a **green** button (treatment) will convert better. They run a 2-week test and report the following results:

| Group | Visitors | Purchases |
|---|---|---|
| Red (control) | 5,000 | 410 |
| Green (treatment) | 5,000 | 460 |

Let's run the full analysis: conversion rates → hypothesis test → confidence interval → business conclusion.

In [ ]:
n_red, conv_red = 5000, 410
n_green, conv_green = 5000, 460

rate_red = conv_red / n_red
rate_green = conv_green / n_green
print(f"Red button conversion rate:   {rate_red:.2%}")
print(f"Green button conversion rate: {rate_green:.2%}")

# Hypothesis test
z_stat, p_val = proportions_ztest(count=[conv_red, conv_green], nobs=[n_red, n_green])
print(f"\nZ-statistic: {z_stat:.3f}, p-value: {p_val:.4f}")
interpret(p_val, 0.05, "Button color test")

# Confidence interval for the difference (green - red)
diff = rate_green - rate_red
se = np.sqrt(rate_red * (1 - rate_red) / n_red + rate_green * (1 - rate_green) / n_green)
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se
print(f"\n95% CI for the difference in conversion rate: [{ci_low:.4f}, {ci_high:.4f}]")

# Business conclusion
relative_lift = (rate_green - rate_red) / rate_red
print(f"\nBusiness takeaway: switching to green gives a {relative_lift:.1%} relative lift in conversions.")
print("Since the result is statistically significant and the effect is meaningful, recommend rolling out the green button.")

### ✍️ Practice 2

Suppose the store had only run the test for 2 days instead of 2 weeks, collecting just **200 visitors per group**, but with the **same conversion rates** (8.2% vs 9.2%, i.e. `conv_red = 16`, `conv_green = 18`).

**Your task:**
1. Re-run the z-test with these smaller numbers.
2. Is the result still statistically significant? Print your conclusion.
3. In a markdown cell (or a comment), explain in your own words *why* the conclusion changed even though the conversion rates are the same.

In [ ]:
# Your code here:

## 14. Business Case: Pricing Strategy Test

**Scenario:** A SaaS company wants to test two pricing pages:
- **Plan A ($29/month)** — control
- **Plan B ($39/month, reframed with more emphasis on features)** — treatment

They measure `monthly_spend`: how much each customer who signed up actually pays per month (a **continuous** metric this time, not binary). Let's simulate 800 customers per group and analyze the results.

In [ ]:
n_per_group = 800

# Simulate monthly spend for customers who signed up under each plan
plan_a_spend = np.random.normal(loc=29, scale=6, size=n_per_group).clip(min=0)
plan_b_spend = np.random.normal(loc=33, scale=7, size=n_per_group).clip(min=0)

print(f"Plan A average monthly spend: ${plan_a_spend.mean():.2f}")
print(f"Plan B average monthly spend: ${plan_b_spend.mean():.2f}")

# Hypothesis test (Welch's t-test)
t_stat, p_val_pricing = stats.ttest_ind(plan_a_spend, plan_b_spend, equal_var=False)
print(f"\nT-statistic: {t_stat:.3f}, p-value: {p_val_pricing:.4f}")
interpret(p_val_pricing, 0.05, "Pricing strategy test")

# Confidence interval for the difference (B - A)
mean_diff = plan_b_spend.mean() - plan_a_spend.mean()
se = np.sqrt(plan_a_spend.var(ddof=1) / n_per_group + plan_b_spend.var(ddof=1) / n_per_group)
ci_low, ci_high = mean_diff - 1.96 * se, mean_diff + 1.96 * se
print(f"\n95% CI for the difference in monthly spend: [${ci_low:.2f}, ${ci_high:.2f}]")

# Effect size
d = cohens_d(pd.Series(plan_a_spend), pd.Series(plan_b_spend))
print(f"Cohen's d: {d:.3f}")

print("\nBusiness takeaway: Plan B customers pay significantly more per month on average.")
print("Note: this only looks at revenue per PAYING customer - a complete decision should also")
print("check whether Plan B has a lower signup/conversion rate, since fewer but bigger-paying")
print("customers isn't automatically better for total revenue.")

### ✍️ Practice 3

The company also tracked whether visitors signed up **at all** (not just how much they paid). Out of 800 visitors shown each pricing page:
- Plan A: 96 signed up
- Plan B: 72 signed up

**Your task:**
1. Calculate the signup (conversion) rate for each plan.
2. Run a two-proportion z-test to check if this difference is statistically significant.
3. Combine this with the `monthly_spend` result above: would you recommend Plan A or Plan B? Write your reasoning in a markdown cell.

In [ ]:
# Your code here:

## 15. Common Pitfalls & Sample Size Considerations

A few mistakes that are very easy to make in real A/B tests:

1. **Peeking**: checking the p-value every day and stopping as soon as it dips below 0.05. This inflates your false-positive rate way beyond 5% — we'll prove this below with a simulation.
2. **Sample Ratio Mismatch (SRM)**: if your control/treatment split isn't close to what you designed (e.g. 50/50), something is broken in your randomization — always check `value_counts()` first (section 3)!
3. **Novelty effect**: users react to *any* change at first, just because it's new. Short tests can show a "fake win" that fades after a week.
4. **Multiple comparisons**: testing many metrics (or many variants) increases the chance that *something* looks significant by pure luck. Correct for this (e.g. Bonferroni correction) when you test more than one thing.
5. **Not calculating sample size in advance**: running a test with too few users means you'll likely miss a real effect (low statistical power).

### Demo: Why "Peeking" is Dangerous

Below we simulate 500 experiments where **both groups have the exact same true conversion rate** (so `H0` is actually true — there is NO real difference). We compare two strategies:
- **Peek**: check the p-value every 100 new users and stop testing as soon as `p < 0.05`.
- **Wait**: only look at the p-value once, after collecting the full planned sample.

If our test is well-behaved, both should give a false-positive rate close to 5%. Let's see what actually happens.

In [ ]:
np.random.seed(1)

n_simulations = 500
max_n = 2000
check_points = np.arange(100, max_n + 1, 100)  # peek every 100 users

false_positive_if_peek = 0
false_positive_if_wait = 0
true_p = 0.10  # SAME rate for both groups -> H0 is actually TRUE

for _ in range(n_simulations):
    control_data = np.random.binomial(1, true_p, max_n)
    treatment_data = np.random.binomial(1, true_p, max_n)

    peeked_significant = False
    for n in check_points:
        c_sub, t_sub = control_data[:n], treatment_data[:n]
        _, p = proportions_ztest([c_sub.sum(), t_sub.sum()], [n, n])
        if p < 0.05:
            peeked_significant = True  # would have stopped the test here and declared a "winner"

    if peeked_significant:
        false_positive_if_peek += 1

    _, p_final = proportions_ztest([control_data.sum(), treatment_data.sum()], [max_n, max_n])
    if p_final < 0.05:
        false_positive_if_wait += 1

print(f"False positive rate if you PEEK repeatedly: {false_positive_if_peek / n_simulations:.1%}")
print(f"False positive rate if you WAIT for the full sample size: {false_positive_if_wait / n_simulations:.1%}")
print("\nRemember: both groups had the SAME true conversion rate, so any 'win' here is a false alarm!")

### Calculating the Required Sample Size (Before Running a Test)

Instead of guessing, we can calculate exactly how many users we need **before** launching the test, given:
- the baseline conversion rate,
- the smallest lift we care about detecting,
- our desired **statistical power** (usually 80%) — the probability of detecting a real effect if one exists.

In [ ]:
# How many users per group do we need to reliably detect a lift from 10% to 12% conversion?
baseline_rate = 0.10
target_rate = 0.12

effect_size = proportion_effectsize(baseline_rate, target_rate)  # Cohen's h

power_analysis = NormalIndPower()
required_n_per_group = power_analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, ratio=1.0)

print(f"Effect size (Cohen's h): {effect_size:.3f}")
print(f"Required sample size PER GROUP: {round(required_n_per_group)}")
print(f"Total users needed: {round(required_n_per_group) * 2}")

## 🎓 Wrap-Up: A/B Testing Cheat Sheet

| Metric type | Example | Test to use | Python function |
|---|---|---|---|
| Binary (yes/no) | conversion, click, signup | Two-proportion z-test | `proportions_ztest()` |
| Continuous | time spent, revenue, spend | Independent t-test | `scipy.stats.ttest_ind()` |
| Effect size (binary) | how big is the rate difference? | Cohen's h | `proportion_effectsize()` |
| Effect size (continuous) | how big is the mean difference? | Cohen's d | our `cohens_d()` function |
| Sample size (binary) | how many users do I need? | Power analysis | `NormalIndPower().solve_power()` |
| Sample size (continuous) | how many users do I need? | Power analysis | `TTestIndPower().solve_power()` |

**The golden workflow**: Formulate hypotheses → Calculate required sample size → Collect data → Check assumptions → Run the test **once** → Interpret p-value + confidence interval + effect size together → Make a business decision.